In [0]:
%restart_python

In [0]:
# 1. Install yfinance directly on your running cluster node
%pip install yfinance

In [0]:
import yfinance as yf
import pandas as pd

# 2. Download historical static market data (Returns a Pandas DataFrame)
# Fetching daily prices for Apple and Microsoft
ticker_symbols = ["AAPL", "MSFT"]
raw_data = yf.download(ticker_symbols, start="2026-01-01", end="2026-11-01")

# 3. Restructure the data layout so Spark can process it smoothly
# This flattens the columns and creates explicit 'Date' and 'Ticker' columns
flat_data = raw_data['Close'].stack().reset_index()

In [0]:
# View the columns of raw_data DataFrame
print(raw_data.columns)
column_list = raw_data.columns.tolist()
print(column_list)
# Push 'Date' from the row index into a regular table column layout
flat_view = raw_data.reset_index()
# Now look at the columns—Date will be part of the structural list!
print(flat_view.columns[:5])
print(flat_view.head())
# Starting with raw multi-index data (Wide structure)
# Columns have 2 levels: Price Type and Ticker
print(raw_data['Close'].head(1))
#Applying stack() 
stacked = raw_data['Close'].stack()
print(stacked.head(2))
stacked_reset_index = raw_data['Close'].stack().reset_index()
print(stacked_reset_index.head(2))

In [0]:
print(raw_data.head())
print(type(raw_data))
print(flat_data.head())
print(type(flat_data))

In [0]:
import yfinance as yf
import pandas as pd

# You are calling the local 'download' API method of the yfinance module
# It handles the background work and hands you back a memory object
#Download historical static market data (Returns a Pandas DataFrame)
# Fetching daily prices for Apple and Microsoft
ticker_symbols = ["AAPL", "MSFT"]
raw_data = yf.download(ticker_symbols, start="2026-01-01", end="2026-11-01")

# Stack the entire DataFrame (this moves the 'Ticker' level down to the rows)
# By default, stack() drops the innermost level (Ticker) to rows, keeping metrics on top
stacked_data = raw_data.stack(level='Ticker')
print(stacked_data.columns)
column_list = stacked_data.columns.tolist()
print(column_list)

#Flatten the Row Index so 'Date' and 'Ticker' become regular columns
flat_all_metrics = stacked_data.reset_index()
print(flat_all_metrics.columns)
column_list = flat_all_metrics.columns.tolist()
print(column_list)
display(flat_all_metrics.head(10))

# Clean up column formatting (convert Date column from Timestamp to explicit String)
flat_all_metrics['Date'] = flat_all_metrics['Date'].dt.strftime('%Y-%m-%d')

#Display the first 10 rows of your complete multi-metric data frame
display(flat_all_metrics.head(10))

In [0]:
import requests

# You are interacting directly with an HTTP REST endpoint using a GET request
# Note: the API lives at /query — the bare domain only returns an HTML homepage
url = "https://www.alphavantage.co/query"
params = {"function": "TIME_SERIES_DAILY", "symbol": "IBM", "apikey": "demo"}
response = requests.get(url, params=params)

# The REST server responds with a JSON text payload
data = response.json()
print(data)

### The difference between an API and a REST API comes down to scope: API is the broad, overarching concept, while a REST API is a specific architectural style of building an API.
- An easy analogy is to think of an API as a vehicle, and a REST API as an electric car. All electric cars are vehicles, but not all vehicles are electric cars.
###Here is a direct comparison table outlining the core technical differences:

|Feature|API (Application Programming Interface)|REST API (REpresentational State Transfer)|
| :--- | :--- | :--- |
|**Definition**|Any interface that allows two software programs to communicate with each other.|A specific type of web API that follows strict design rules over HTTP.|
|**Communication Medium**|Can use local memory, files, commands, networks, or system calls.|Must use network protocols, specifically HTTP/HTTPS.|
|**Data Formats**|Can use any structure (Objects, arrays, binary files, XML, raw text).|Primarily uses JSON (preferred) or XML payloads.|
|**Statelessness**|Can be stateful (remembers your session) or stateless.|Must be completely stateless (every single request must contain all its own authentication/context).|
|**Examples**|yfinance library functions, Python’s dict.get(), database drivers.|Yahoo Finance's web endpoint, Alpha Vantage URL endpoints, Bloomberg WAPI.|

### 1. The Broad Concept: API
An API is any set of rules or code blocks that allows one piece of software to use the features or data of another piece of software. It does not require a network or the internet.
- How it works locally: When you import a library like yfinance or use scores.get() in Python, you are interacting with an API. The code executes directly inside your computer's local memory loop.
- No Web Needed: The communication happens entirely within your operating system or application runtime.

### 2. The Specific Standard: REST API
A REST API is an API designed specifically for the web. It uses standard HTTP actions to modify or retrieve data from a remote server over the internet.

It enforces a specific design philosophy called REST:
- HTTP Methods: It requires you to use standard web actions: 
    - GET to read data
    - POST to create data
    - PUT to update data
    - DELETE to remove data
- Endpoints (URLs): Data is treated as a "resource" located at a specific web address (e.g., https://provider.com).
- JSON Payloads: Because it runs over the web, it almost always wraps its data inside a standardized, lightweight text format called JSON (which maps directly to Python dictionaries).

###REST stands for Representational State Transfer.
####The words in REST describe how data is handled over a network connection:

1. Representational
> When you request data from a server, you never get the actual live database record itself. Instead, the server sends you a representation of that object. This representation is usually formatted as a clean text file, like a JSON structure or an XML document.

2. State
> In computer science, "State" refers to the current condition, values, or settings of your data at an exact moment in time (for example, a stock's open, high, and close values for today).

3. Transfer
> This refers to moving that snapshot of data over the internet or a network from the server's machine to your local Python workspace.

- The Intuition	--The Technical REST Equivalent (HTTP Method)	    --What It Actually Does
> Store	        --POST / PUT	                                    --Creates or saves a new data record on the server.
>> Read	        --GET	                                            --Retrieves and views a data representation (like your requests.get() code).
>>> Edit	    --PATCH / PUT	                                    --Modifies or updates an existing data record.
>>>> Terminate	--DELETE	                                        --Removes a record completely from the remote database server.